## **Farmland Birds Under Pressure**
# Biodiversity Trends and Agricultural Change in Europe

**Main question**

**How have common farmland-bird populations changed across European countries since 2000, and are agricultural intensity, organic farming and protected-area coverage associated with different national trajectories?**

**Layer 1:** Long-term biodiversity trends

Use:

- env_bio2: national farmland-bird indices
- env_bio3: EU bird-group indices and uncertainty

Source: https://ec.europa.eu/eurostat/web/main/data/database#Updates


**Period:**

1990–2019 for Germany and long-term countries
2000–2019 for the main cross-country comparison
Later years only in a clearly labelled supplementary analysis

**Questions:**

- Which countries experienced the largest decline since 2000?
- Did Germany decline faster than the EU benchmark?
- Are farmland birds declining faster than forest birds?
- Are there countries showing stabilization or recovery?
- How sensitive are rankings to the chosen endpoint?

 **Bird groups**
CO_ALL: all common bird species
CO_FARM: common farmland bird species
CO_FOR: common forest bird species

**Estimate types**
NSME: unsmoothed estimate
SME: smoothed estimate
SME_LW95: lower 95% confidence limit
SME_UP95: upper 95% confidence limit

 **Index reference systems**
I00: 2000 = 100
I90: 1990 = 100
I_LY: latest year = 100


**Layer 2: Environmental drivers**
Add three Eurostat indicators.

1. Organic farming

Dataset:[Area under organic farming — sdg_02_40](https://ec.europa.eu/eurostat/databrowser/view/tag00025/default/table?lang=en)

## 1. Load the Eurostat bulk-download files

Eurostat's TSV files are compressed with gzip. Pandas can read them directly; manual decompression is unnecessary. The path resolver below works both next to the notebook and in the original Google Drive folder.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


def find_data_file(filename):
    # Return the first existing copy of a source file.
    candidates = [
        Path(filename),
        Path("upload") / filename,
        Path("/content/drive/MyDrive/Final_Project") / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    searched = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")


source_paths = {
    "env_bio2": find_data_file("estat_env_bio2.tsv.gz"),
    "env_bio3": find_data_file("estat_env_bio3.tsv.gz"),
    "env_bio4": find_data_file("estat_env_bio4.tsv.gz"),
}

raw_bio2 = pd.read_csv(source_paths["env_bio2"], sep="	", compression="gzip", dtype=str)
raw_bio3 = pd.read_csv(source_paths["env_bio3"], sep="	", compression="gzip", dtype=str)
raw_bio4 = pd.read_csv(source_paths["env_bio4"], sep="	", compression="gzip", dtype=str)

pd.DataFrame(
    {
        "dataset": source_paths.keys(),
        "rows": [len(raw_bio2), len(raw_bio3), len(raw_bio4)],
        "columns": [raw_bio2.shape[1], raw_bio3.shape[1], raw_bio4.shape[1]],
        "source_file": [str(path) for path in source_paths.values()],
    }
)


## 2. Parse Eurostat dimensions, values, and flags

The first column is not one variable. It contains several comma-separated dimensions, while `\TIME_PERIOD` marks the boundary between those dimensions and the year columns.

The raw missing-value check can be misleading: Eurostat represents missing observations with `:` rather than a conventional null. Cells can also contain a number followed by a status flag, for example `60.84 d`. The function below:

1. separates the first column into named dimensions;
2. reshapes years from columns into rows;
3. extracts the numeric observation;
4. keeps Eurostat's status flag in a separate column; and
5. converts `:` to a true missing value.


In [ ]:
def eurostat_to_long(raw_df, dimension_names, dataset_name):
    # Convert a Eurostat bulk TSV table to tidy long format.
    id_column = raw_df.columns[0]
    expected_dimensions = len(dimension_names)

    dimensions = raw_df[id_column].str.split(",", expand=True)
    if dimensions.shape[1] != expected_dimensions:
        raise ValueError(
            f"{dataset_name}: expected {expected_dimensions} dimensions "
            f"but found {dimensions.shape[1]}"
        )
    dimensions.columns = dimension_names

    separated = pd.concat(
        [dimensions, raw_df.drop(columns=id_column)],
        axis=1,
    )

    tidy = separated.melt(
        id_vars=dimension_names,
        var_name="year",
        value_name="raw_value",
    )

    tidy["year"] = pd.to_numeric(tidy["year"].str.strip(), errors="raise").astype("int64")
    tidy["raw_value"] = tidy["raw_value"].astype("string").str.strip()

    extracted = tidy["raw_value"].str.extract(
        r"^(?:([-+]?\d+(?:\.\d+)?)|:)\s*([A-Za-z]+)?$"
    )
    tidy["value"] = pd.to_numeric(extracted[0], errors="coerce")
    tidy["flag"] = extracted[1].astype("string")
    tidy["dataset"] = dataset_name

    unexpected = tidy.loc[
        ~tidy["raw_value"].str.match(
            r"^(?:([-+]?\d+(?:\.\d+)?)|:)\s*([A-Za-z]+)?$",
            na=False,
        ),
        "raw_value",
    ].drop_duplicates()
    if not unexpected.empty:
        raise ValueError(f"{dataset_name}: unparsed cells: {unexpected.tolist()}")

    return tidy[
        ["dataset", *dimension_names, "year", "value", "flag", "raw_value"]
    ].sort_values([*dimension_names, "year"], ignore_index=True)


bio2_long = eurostat_to_long(
    raw_bio2,
    ["frequency", "unit", "country"],
    "env_bio2",
)

bio3_long = eurostat_to_long(
    raw_bio3,
    ["frequency", "estimate_type", "bird_group", "unit", "country"],
    "env_bio3",
)

bio4_long = eurostat_to_long(
    raw_bio4,
    ["frequency", "unit", "protected_area_type", "country"],
    "env_bio4",
)

bio2_long.head()


## 3. Validate the tidy datasets

Duplicates should be checked using the dimensions that uniquely identify an observation, not across the original wide rows. Missing observations should be counted after `:` has been converted to `NaN`.


In [ ]:
def dataset_audit(df, key_columns):
    return {
        "rows": len(df),
        "valid_values": int(df["value"].notna().sum()),
        "missing_values": int(df["value"].isna().sum()),
        "flagged_values": int(df["flag"].notna().sum()),
        "duplicate_keys": int(df.duplicated(key_columns).sum()),
        "first_year": int(df["year"].min()),
        "last_year": int(df["year"].max()),
    }


audit = pd.DataFrame.from_dict(
    {
        "env_bio2": dataset_audit(
            bio2_long, ["frequency", "unit", "country", "year"]
        ),
        "env_bio3": dataset_audit(
            bio3_long,
            ["frequency", "estimate_type", "bird_group", "unit", "country", "year"],
        ),
        "env_bio4": dataset_audit(
            bio4_long,
            ["frequency", "unit", "protected_area_type", "country", "year"],
        ),
    },
    orient="index",
).rename_axis("dataset").reset_index()

audit


## 4. Select the variables needed for the study

### National farmland-bird series (`env_bio2`)

This dataset is already restricted to the annual index with 2000 as its reference. Missing observations are removed from the analytical subset, but flagged observations are retained so that sensitivity checks remain possible.

### EU benchmark (`env_bio3`)

We select common farmland species and the 2000-based index. The unsmoothed estimate, smoothed estimate, and its lower and upper 95% confidence limits are pivoted into separate columns. This produces one EU row per year.

### Protected areas (`env_bio4`)

We use the percentage of terrestrial protected area. Marine protected areas and square-kilometre totals are not appropriate predictors for a national farmland-bird model.


In [ ]:
birds_country = (
    bio2_long.loc[
        bio2_long["frequency"].eq("A")
        & bio2_long["unit"].eq("I00")
        & bio2_long["value"].notna(),
        ["country", "year", "value", "flag"],
    ]
    .rename(columns={"value": "bird_index", "flag": "bird_flag"})
    .sort_values(["country", "year"], ignore_index=True)
)

estimate_names = {
    "NSME": "eu_unsmoothed",
    "SME": "eu_smoothed",
    "SME_LW95": "eu_lower_95",
    "SME_UP95": "eu_upper_95",
}

eu_bird_benchmark = (
    bio3_long.loc[
        bio3_long["frequency"].eq("A")
        & bio3_long["country"].eq("EU27_2020")
        & bio3_long["unit"].eq("I00")
        & bio3_long["bird_group"].eq("CO_FARM")
        & bio3_long["estimate_type"].isin(estimate_names)
        & bio3_long["value"].notna(),
        ["year", "estimate_type", "value"],
    ]
    .pivot(index="year", columns="estimate_type", values="value")
    .rename(columns=estimate_names)
    .rename_axis(columns=None)
    .reset_index()
    .sort_values("year", ignore_index=True)
)

protected_terrestrial = (
    bio4_long.loc[
        bio4_long["frequency"].eq("A")
        & bio4_long["unit"].eq("PC")
        & bio4_long["protected_area_type"].eq("TPA")
        & bio4_long["country"].ne("EU27_2020")
        & bio4_long["value"].notna(),
        ["country", "year", "value", "flag"],
    ]
    .rename(
        columns={
            "value": "protected_terrestrial_pct",
            "flag": "protected_area_flag",
        }
    )
    .sort_values(["country", "year"], ignore_index=True)
)

selection_summary = pd.DataFrame(
    {
        "dataframe": ["birds_country", "eu_bird_benchmark", "protected_terrestrial"],
        "rows": [len(birds_country), len(eu_bird_benchmark), len(protected_terrestrial)],
        "first_year": [
            birds_country["year"].min(),
            eu_bird_benchmark["year"].min(),
            protected_terrestrial["year"].min(),
        ],
        "last_year": [
            birds_country["year"].max(),
            eu_bird_benchmark["year"].max(),
            protected_terrestrial["year"].max(),
        ],
    }
)

selection_summary


## 5. Merge compatible information

The EU benchmark is joined by year. Protected-area coverage is joined by country and year. A left join preserves every valid national bird observation and makes unavailable predictors visible instead of silently deleting those rows.

The resulting `bird_panel` is useful for descriptive comparisons. `model_panel_observed` is the complete-case subset for models that require protected-area coverage. No values are interpolated or filled at this stage.


In [ ]:
bird_panel = (
    birds_country
    .merge(
        eu_bird_benchmark,
        on="year",
        how="left",
        validate="many_to_one",
    )
    .merge(
        protected_terrestrial,
        on=["country", "year"],
        how="left",
        validate="one_to_one",
    )
    .sort_values(["country", "year"], ignore_index=True)
)

bird_panel["bird_minus_eu"] = (
    bird_panel["bird_index"] - bird_panel["eu_smoothed"]
)
bird_panel["bird_change_pct"] = (
    bird_panel.groupby("country")["bird_index"].pct_change(fill_method=None) * 100
)

model_panel_observed = (
    bird_panel.dropna(
        subset=["bird_index", "eu_smoothed", "protected_terrestrial_pct"]
    )
    .reset_index(drop=True)
)

merge_audit = pd.Series(
    {
        "national bird rows": len(bird_panel),
        "rows with EU benchmark": int(bird_panel["eu_smoothed"].notna().sum()),
        "rows with protected-area value": int(
            bird_panel["protected_terrestrial_pct"].notna().sum()
        ),
        "complete rows for current model": len(model_panel_observed),
        "countries in current model": model_panel_observed["country"].nunique(),
    },
    name="count",
).to_frame()

merge_audit


## 6. Create country-level trend features

The primary comparison window is 2000–2019 because it preserves a common reference year and includes Germany's latest available bird observation. For each country with sufficient observations, the table below calculates:

- the first and last available index in the window;
- total percentage change;
- an ordinary least-squares slope in index points per year;
- coverage and flag counts; and
- the country's 2019 position relative to the EU smoothed index.

The slope summarizes direction; it should not be interpreted as a population forecast.


In [ ]:
def summarize_country_trend(group, start_year=2000, end_year=2019, min_years=10):
    sample = group.loc[
        group["year"].between(start_year, end_year),
        ["year", "bird_index", "bird_flag"],
    ].dropna(subset=["bird_index"]).sort_values("year")

    if len(sample) < min_years:
        return pd.Series(
            {
                "first_year": pd.NA,
                "last_year": pd.NA,
                "first_index": np.nan,
                "last_index": np.nan,
                "change_pct": np.nan,
                "slope_points_per_year": np.nan,
                "n_years": len(sample),
                "flagged_years": int(sample["bird_flag"].notna().sum()),
            }
        )

    first = sample.iloc[0]
    last = sample.iloc[-1]
    slope = np.polyfit(sample["year"], sample["bird_index"], 1)[0]

    return pd.Series(
        {
            "first_year": int(first["year"]),
            "last_year": int(last["year"]),
            "first_index": first["bird_index"],
            "last_index": last["bird_index"],
            "change_pct": (last["bird_index"] / first["bird_index"] - 1) * 100,
            "slope_points_per_year": slope,
            "n_years": len(sample),
            "flagged_years": int(sample["bird_flag"].notna().sum()),
        }
    )


country_trends_2000_2019 = (
    birds_country.groupby("country")[["year", "bird_index", "bird_flag"]]
    .apply(summarize_country_trend)
    .reset_index()
    .dropna(subset=["change_pct"])
    .sort_values("change_pct", ignore_index=True)
)

country_trends_2000_2019


## 7. Initial visual checks

The first plot compares Germany with the EU smoothed farmland-bird index and its 95% confidence interval. The second plot ranks national changes over the common 2000–2019 study period.


In [ ]:
germany = birds_country.query("country == 'DE'")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(germany["year"], germany["bird_index"], label="Germany", linewidth=2.3)
ax.plot(
    eu_bird_benchmark["year"],
    eu_bird_benchmark["eu_smoothed"],
    label="EU-27 smoothed estimate",
    linewidth=2.3,
)
ax.fill_between(
    eu_bird_benchmark["year"],
    eu_bird_benchmark["eu_lower_95"],
    eu_bird_benchmark["eu_upper_95"],
    alpha=0.18,
    label="EU-27 95% confidence interval",
)
ax.axhline(100, color="grey", linestyle="--", linewidth=1, label="2000 reference")
ax.set(title="Common farmland bird index", xlabel="Year", ylabel="Index (2000 = 100)")
ax.legend(frameon=False, ncols=2)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
plot_data = country_trends_2000_2019.sort_values("change_pct")

fig, ax = plt.subplots(figsize=(10, 7))
colors = np.where(plot_data["change_pct"] < 0, "#9F3A38", "#2A7F62")
ax.barh(plot_data["country"], plot_data["change_pct"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(
    title="Change in national common farmland bird indices",
    xlabel="Change from first to last available observation, 2000–2019 (%)",
    ylabel="Eurostat country code",
)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## 8. Export analysis-ready DataFrames

The exports keep the data at clearly defined grains:

- `birds_country_long.csv`: one national bird index per country-year;
- `eu_farmland_bird_benchmark.csv`: one EU benchmark row per year;
- `protected_terrestrial_long.csv`: one protected-area percentage per country-year;
- `bird_country_year_panel.csv`: merged descriptive panel, including missing predictors;
- `bird_model_panel_observed.csv`: complete rows for the current protected-area model;
- `country_trends_2000_2019.csv`: country-level trend summary; and
- `data_quality_audit.csv`: parsing and quality-control counts.


In [ ]:
output_dir = Path("outputs/dataframes")
output_dir.mkdir(parents=True, exist_ok=True)

exports = {
    "birds_country_long.csv": birds_country,
    "eu_farmland_bird_benchmark.csv": eu_bird_benchmark,
    "protected_terrestrial_long.csv": protected_terrestrial,
    "bird_country_year_panel.csv": bird_panel,
    "bird_model_panel_observed.csv": model_panel_observed,
    "country_trends_2000_2019.csv": country_trends_2000_2019,
    "data_quality_audit.csv": audit,
}

for filename, dataframe in exports.items():
    dataframe.to_csv(output_dir / filename, index=False)

export_summary = pd.DataFrame(
    {
        "file": exports.keys(),
        "rows": [len(df) for df in exports.values()],
        "columns": [df.shape[1] for df in exports.values()],
    }
)

export_summary


## 9. Recommended next stage

The present panel can describe national bird trajectories and their relationship with protected-area coverage. It is not yet a robust agricultural-pressure model because protected-area percentage changes slowly and is only available from 2011.

The next additions should be reshaped with the same function and merged on `country` and `year`:

1. `sdg_02_40`: percentage of utilised agricultural area under organic farming;
2. `aei_pr_gnb`: nitrogen balance per hectare of utilised agricultural area; and
3. optionally `aei_fm_salpest09`: pesticide sales, normalized by utilised agricultural area.

Before fitting a model:

- measure country-year overlap after every join;
- keep flags rather than discarding them silently;
- avoid treating an index as an absolute bird count;
- compare a country-and-year baseline with models containing environmental indicators;
- use a time-based or country-grouped validation split rather than a random row split; and
- describe model results as associations, not causal effects.


## 10. Add agricultural indicators from the Eurostat API

The predictive part uses three environmental indicators:

- **Organic farming:** percentage of utilised agricultural area under organic management (`sdg_02_40`).
- **Pesticide sales:** total kilograms sold (`aei_fm_salpest09`).
- **Utilised agricultural area:** thousand hectares (`tag00025`), used to convert pesticide sales into kilograms per hectare.

Pesticide totals should not be compared directly between large and small countries. The derived intensity is:

\[
	ext{pesticide intensity} =
rac{	ext{pesticide sales in kg}}
{	ext{utilised agricultural area in thousand ha} 	imes 1000}
\]

The function below reads Eurostat's JSON-stat response using only Python's standard library, so it also works in environments where `requests` is unavailable.


In [ ]:
import json
import urllib.parse
import urllib.request


EUROSTAT_API = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"


def fetch_eurostat_jsonstat(dataset_code, filters=None, timeout=90):
    # Fetch a filtered Eurostat dataset and return observations in long format.
    params = {"lang": "en"}
    if filters:
        params.update(filters)
    url = f"{EUROSTAT_API}/{dataset_code}?{urllib.parse.urlencode(params)}"

    with urllib.request.urlopen(url, timeout=timeout) as response:
        payload = json.load(response)

    dimensions = payload["id"]
    sizes = payload["size"]
    dimension_codes = []

    for dimension in dimensions:
        category_index = payload["dimension"][dimension]["category"]["index"]
        if isinstance(category_index, dict):
            ordered_codes = [
                code_value
                for code_value, _ in sorted(category_index.items(), key=lambda item: item[1])
            ]
        else:
            ordered_codes = list(category_index)
        dimension_codes.append(ordered_codes)

    observations = []
    statuses = payload.get("status", {})
    for flat_index_text, value in payload.get("value", {}).items():
        flat_index = int(flat_index_text)
        remaining = flat_index
        coordinates = []
        for size in reversed(sizes):
            coordinates.append(remaining % size)
            remaining //= size
        coordinates.reverse()

        row = {
            dimension: dimension_codes[position][coordinate]
            for position, (dimension, coordinate) in enumerate(
                zip(dimensions, coordinates)
            )
        }
        row["value"] = value
        row["flag"] = statuses.get(flat_index_text, pd.NA)
        observations.append(row)

    result = pd.DataFrame(observations)
    result.attrs["dataset_code"] = dataset_code
    result.attrs["dataset_label"] = payload.get("label", dataset_code)
    result.attrs["source_url"] = url
    return result


organic_raw = fetch_eurostat_jsonstat(
    "sdg_02_40",
    {
        "unit": "PC_UAA",
        "crops": "UAAXK0000",
        "agprdmet": "TOTAL",
    },
)

pesticide_raw = fetch_eurostat_jsonstat(
    "aei_fm_salpest09",
    {"pesticid": "TOTAL", "unit": "KG"},
)

uaa_raw = fetch_eurostat_jsonstat(
    "tag00025",
    {"crops": "UAA", "strucpro": "MAR_THS_HA"},
)

pd.DataFrame(
    {
        "dataset": ["sdg_02_40", "aei_fm_salpest09", "tag00025"],
        "meaning": [
            "Organic farming share",
            "Total pesticide sales",
            "Utilised agricultural area",
        ],
        "retrieved_rows": [len(organic_raw), len(pesticide_raw), len(uaa_raw)],
    }
)


## 11. Clean and normalize the agricultural indicators

All three API tables use Eurostat country codes. They can therefore be joined to the bird data using `country` and `year`. Aggregate areas such as the EU total are removed because the predictive observations are countries.

The status flags are kept in separate columns. Pesticide intensity is only calculated when both pesticide sales and agricultural area are positive.


In [ ]:
AGGREGATE_GEOS = {
    "EU27_2020", "EU28", "EA19", "EA20", "EFTA", "EEA30_2007",
    "CC4_2020", "CC3_2007",
}


def prepare_api_indicator(raw, value_name, flag_name):
    prepared = raw.rename(
        columns={"geo": "country", "time": "year", "value": value_name, "flag": flag_name}
    ).copy()
    prepared["year"] = pd.to_numeric(prepared["year"], errors="raise").astype("int64")
    prepared[value_name] = pd.to_numeric(prepared[value_name], errors="coerce")
    prepared = prepared.loc[
        ~prepared["country"].isin(AGGREGATE_GEOS),
        ["country", "year", value_name, flag_name],
    ]
    return prepared.sort_values(["country", "year"], ignore_index=True)


organic_farming = prepare_api_indicator(
    organic_raw, "organic_farming_pct", "organic_flag"
)
pesticide_sales = prepare_api_indicator(
    pesticide_raw, "pesticide_sales_kg", "pesticide_flag"
)
agricultural_area = prepare_api_indicator(
    uaa_raw, "uaa_thousand_ha", "uaa_flag"
)

pesticide_intensity = pesticide_sales.merge(
    agricultural_area,
    on=["country", "year"],
    how="inner",
    validate="one_to_one",
)
pesticide_intensity["pesticide_kg_per_ha"] = (
    pesticide_intensity["pesticide_sales_kg"]
    / (pesticide_intensity["uaa_thousand_ha"] * 1000)
)
pesticide_intensity.loc[
    pesticide_intensity["uaa_thousand_ha"].le(0), "pesticide_kg_per_ha"
] = np.nan

indicator_audit = pd.DataFrame(
    {
        "dataframe": ["organic_farming", "pesticide_sales", "agricultural_area", "pesticide_intensity"],
        "rows": [len(organic_farming), len(pesticide_sales), len(agricultural_area), len(pesticide_intensity)],
        "countries": [
            organic_farming["country"].nunique(),
            pesticide_sales["country"].nunique(),
            agricultural_area["country"].nunique(),
            pesticide_intensity["country"].nunique(),
        ],
        "first_year": [
            organic_farming["year"].min(),
            pesticide_sales["year"].min(),
            agricultural_area["year"].min(),
            pesticide_intensity["year"].min(),
        ],
        "last_year": [
            organic_farming["year"].max(),
            pesticide_sales["year"].max(),
            agricultural_area["year"].max(),
            pesticide_intensity["year"].max(),
        ],
    }
)

indicator_audit


## 12. Build the machine-learning panel and target

### Prediction problem

For each country and year \(t\), predict the farmland-bird index in year \(t+1\).

The predictors are information available at year \(t\):

- current national bird index;
- current one-year change in the national index;
- EU smoothed farmland-bird index;
- organic-farming percentage;
- pesticide sales per hectare;
- terrestrial protected-area percentage;
- country and year.

Including the current bird index makes the task realistic but creates a strong persistence baseline. A useful model must improve upon the simple prediction:

\[
\widehat{BirdIndex}_{t+1}=BirdIndex_t
\]

Only consecutive years are used. A missing observation in between years is not treated as a one-year transition.


In [ ]:
ml_panel = (
    bird_panel
    .merge(
        organic_farming[["country", "year", "organic_farming_pct", "organic_flag"]],
        on=["country", "year"], how="left", validate="one_to_one",
    )
    .merge(
        pesticide_intensity[
            [
                "country", "year", "pesticide_sales_kg", "uaa_thousand_ha",
                "pesticide_kg_per_ha", "pesticide_flag", "uaa_flag",
            ]
        ],
        on=["country", "year"], how="left", validate="one_to_one",
    )
    .sort_values(["country", "year"], ignore_index=True)
)

ml_panel["next_year"] = ml_panel.groupby("country")["year"].shift(-1)
ml_panel["target_next_bird_index"] = ml_panel.groupby("country")["bird_index"].shift(-1)
ml_panel.loc[
    ml_panel["next_year"].ne(ml_panel["year"] + 1),
    "target_next_bird_index",
] = np.nan

ml_panel["decline_next_year"] = (
    ml_panel["target_next_bird_index"] < ml_panel["bird_index"]
).where(ml_panel["target_next_bird_index"].notna())

feature_columns = [
    "year",
    "country",
    "bird_index",
    "bird_change_pct",
    "eu_smoothed",
    "organic_farming_pct",
    "pesticide_kg_per_ha",
    "protected_terrestrial_pct",
]

required_for_model = feature_columns + ["target_next_bird_index"]
model_data = (
    ml_panel.dropna(subset=required_for_model)
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=required_for_model)
    .reset_index(drop=True)
)
model_data["decline_next_year"] = model_data["decline_next_year"].astype("int8")

model_data_summary = pd.Series(
    {
        "country-year rows": len(model_data),
        "countries": model_data["country"].nunique(),
        "first predictor year": model_data["year"].min(),
        "last predictor year": model_data["year"].max(),
        "decline observations": int(model_data["decline_next_year"].sum()),
        "non-decline observations": int((1 - model_data["decline_next_year"]).sum()),
    },
    name="value",
).to_frame()

model_data_summary


## 13. Train/test design and preprocessing

A random row split would leak temporal and country-specific information because neighbouring observations from the same country are highly related.

- **Final holdout:** predictor years from 2019 onward.
- **Training set:** years before 2019.
- **Cross-validation:** `GroupKFold`, grouping rows by country during hyperparameter tuning.

Numeric features are median-imputed and standardized. Country is most-frequent-imputed and one-hot encoded. Tree models do not require scaling, but using the same preprocessing interface makes model comparison and deployment consistent.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GroupKFold, GridSearchCV


TEST_START_YEAR = 2019

train_data = model_data.loc[model_data["year"] < TEST_START_YEAR].copy()
test_data = model_data.loc[model_data["year"] >= TEST_START_YEAR].copy()

X_train = train_data[feature_columns]
X_test = test_data[feature_columns]
y_train_reg = train_data["target_next_bird_index"]
y_test_reg = test_data["target_next_bird_index"]
y_train_cls = train_data["decline_next_year"]
y_test_cls = test_data["decline_next_year"]

numeric_features = [column for column in feature_columns if column != "country"]
categorical_features = ["country"]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("country", categorical_pipeline, categorical_features),
    ]
)

n_country_groups = train_data["country"].nunique()
group_cv = GroupKFold(n_splits=min(5, n_country_groups))

split_summary = pd.DataFrame(
    {
        "partition": ["Training", "Final time holdout"],
        "rows": [len(train_data), len(test_data)],
        "countries": [train_data["country"].nunique(), test_data["country"].nunique()],
        "first_year": [train_data["year"].min(), test_data["year"].min()],
        "last_year": [train_data["year"].max(), test_data["year"].max()],
    }
)

split_summary


## 14. Regression models and hyperparameter tuning

Three predictions are compared:

1. **Persistence baseline:** next year's index equals the current index.
2. **Ridge regression:** a regularized, interpretable linear model.
3. **Random forest:** a nonlinear ensemble model.

Mean absolute error (MAE) is the primary metric because it remains in bird-index points and is easy to communicate. RMSE penalizes large errors more strongly, while \(R^2\) measures improvement relative to predicting the test-set mean. Negative \(R^2\) means the model performs worse than that mean baseline.

Hyperparameters are selected using grouped cross-validation on the training set. The final time holdout is not used for tuning.


In [ ]:
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


ridge_pipeline = Pipeline(
    steps=[("preprocess", clone(preprocessor)), ("model", Ridge())]
)
ridge_search = GridSearchCV(
    ridge_pipeline,
    param_grid={"model__alpha": [0.1, 1.0, 10.0, 100.0]},
    scoring="neg_mean_absolute_error",
    cv=group_cv,
    n_jobs=-1,
    refit=True,
)
ridge_search.fit(X_train, y_train_reg, groups=train_data["country"])

forest_pipeline = Pipeline(
    steps=[
        ("preprocess", clone(preprocessor)),
        ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
    ]
)
forest_search = GridSearchCV(
    forest_pipeline,
    param_grid={
        "model__n_estimators": [200, 500],
        "model__max_depth": [3, None],
        "model__min_samples_leaf": [2, 5],
        "model__max_features": ["sqrt", 0.8],
    },
    scoring="neg_mean_absolute_error",
    cv=group_cv,
    n_jobs=-1,
    refit=True,
)
forest_search.fit(X_train, y_train_reg, groups=train_data["country"])


def regression_metrics(name, y_true, prediction, cv_mae=np.nan):
    return {
        "model": name,
        "cv_mae": cv_mae,
        "test_mae": mean_absolute_error(y_true, prediction),
        "test_rmse": mean_squared_error(y_true, prediction) ** 0.5,
        "test_r2": r2_score(y_true, prediction),
    }


regression_predictions = {
    "Persistence baseline": X_test["bird_index"].to_numpy(),
    "Ridge regression": ridge_search.predict(X_test),
    "Random forest": forest_search.predict(X_test),
}

regression_results = pd.DataFrame(
    [
        regression_metrics(
            "Persistence baseline",
            y_test_reg,
            regression_predictions["Persistence baseline"],
        ),
        regression_metrics(
            "Ridge regression",
            y_test_reg,
            regression_predictions["Ridge regression"],
            -ridge_search.best_score_,
        ),
        regression_metrics(
            "Random forest",
            y_test_reg,
            regression_predictions["Random forest"],
            -forest_search.best_score_,
        ),
    ]
).sort_values("test_mae", ignore_index=True)

regression_results


In [ ]:
regression_tuning = pd.DataFrame(
    {
        "model": ["Ridge regression", "Random forest"],
        "best_grouped_cv_mae": [-ridge_search.best_score_, -forest_search.best_score_],
        "best_parameters": [ridge_search.best_params_, forest_search.best_params_],
    }
)

# Choose the final model using cross-validation, not the final test scores.
regression_candidates = {
    "Ridge regression": ridge_search,
    "Random forest": forest_search,
}
best_regression_name = regression_tuning.sort_values("best_grouped_cv_mae").iloc[0]["model"]
best_regression_model = regression_candidates[best_regression_name].best_estimator_

train_prediction = best_regression_model.predict(X_train)
test_prediction = best_regression_model.predict(X_test)

bias_variance_check = pd.DataFrame(
    {
        "measurement": ["Training MAE", "Grouped CV MAE", "Time-holdout MAE"],
        "mae": [
            mean_absolute_error(y_train_reg, train_prediction),
            regression_tuning.set_index("model").loc[best_regression_name, "best_grouped_cv_mae"],
            mean_absolute_error(y_test_reg, test_prediction),
        ],
    }
)

print("Selected from grouped cross-validation:", best_regression_name)
display(regression_tuning)
display(bias_variance_check)


## 15. Regression diagnostics and feature importance

The prediction plot reveals systematic under- or over-prediction. Permutation importance measures how much the holdout MAE deteriorates when each original feature is shuffled. This is easier to compare across linear and tree models than raw coefficients or split importance.

Feature importance is predictive rather than causal. A strong country or year effect does not identify an ecological mechanism.


In [ ]:
from sklearn.inspection import permutation_importance


test_predictions = test_data[
    ["country", "year", "bird_index", "target_next_bird_index"]
].copy()
test_predictions["predicted_next_bird_index"] = test_prediction
test_predictions["residual"] = (
    test_predictions["target_next_bird_index"]
    - test_predictions["predicted_next_bird_index"]
)

importance_result = permutation_importance(
    best_regression_model,
    X_test,
    y_test_reg,
    scoring="neg_mean_absolute_error",
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
)
regression_feature_importance = (
    pd.DataFrame(
        {
            "feature": feature_columns,
            "importance_mean": importance_result.importances_mean,
            "importance_std": importance_result.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False, ignore_index=True)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test_reg, test_prediction, alpha=0.75)
limits = [min(y_test_reg.min(), test_prediction.min()), max(y_test_reg.max(), test_prediction.max())]
axes[0].plot(limits, limits, "--", color="black", linewidth=1)
axes[0].set(
    title=f"Time-holdout predictions: {best_regression_name}",
    xlabel="Observed next-year bird index",
    ylabel="Predicted next-year bird index",
)

importance_plot = regression_feature_importance.sort_values("importance_mean")
axes[1].barh(
    importance_plot["feature"],
    importance_plot["importance_mean"],
    xerr=importance_plot["importance_std"],
    color="#486A8C",
)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set(
    title="Permutation importance on the time holdout",
    xlabel="Increase in MAE after permutation",
)

plt.tight_layout()
plt.show()

regression_feature_importance


## 16. Secondary classification task: will the index decline next year?

The target is `1` when the next-year bird index is lower than the current index. Because decline and non-decline classes are not equally frequent, accuracy alone can be misleading.

The principal classification metrics are:

- **F1 score:** balance between precision and recall for decline years;
- **balanced accuracy:** average recall across both classes;
- **precision:** share of predicted declines that were declines;
- **recall:** share of actual declines identified; and
- **ROC AUC:** ranking quality across probability thresholds.

Class-weighted logistic regression and random forest models are compared with a most-frequent dummy classifier.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


dummy_classifier = Pipeline(
    steps=[
        ("preprocess", clone(preprocessor)),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]
)
dummy_classifier.fit(X_train, y_train_cls)

logistic_pipeline = Pipeline(
    steps=[
        ("preprocess", clone(preprocessor)),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=3000,
                random_state=42,
            ),
        ),
    ]
)
logistic_search = GridSearchCV(
    logistic_pipeline,
    param_grid={"model__C": [0.01, 0.1, 1.0, 10.0]},
    scoring="f1",
    cv=group_cv,
    n_jobs=-1,
    refit=True,
)
logistic_search.fit(X_train, y_train_cls, groups=train_data["country"])

classifier_forest_pipeline = Pipeline(
    steps=[
        ("preprocess", clone(preprocessor)),
        (
            "model",
            RandomForestClassifier(
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)
classifier_forest_search = GridSearchCV(
    classifier_forest_pipeline,
    param_grid={
        "model__n_estimators": [200, 500],
        "model__max_depth": [3, None],
        "model__min_samples_leaf": [2, 5],
        "model__max_features": ["sqrt", 0.8],
    },
    scoring="f1",
    cv=group_cv,
    n_jobs=-1,
    refit=True,
)
classifier_forest_search.fit(X_train, y_train_cls, groups=train_data["country"])


def classification_metrics(name, model, X, y, cv_f1=np.nan):
    prediction = model.predict(X)
    probability = model.predict_proba(X)[:, 1]
    return {
        "model": name,
        "cv_f1": cv_f1,
        "test_f1": f1_score(y, prediction, zero_division=0),
        "test_balanced_accuracy": balanced_accuracy_score(y, prediction),
        "test_precision": precision_score(y, prediction, zero_division=0),
        "test_recall": recall_score(y, prediction, zero_division=0),
        "test_roc_auc": roc_auc_score(y, probability),
    }


classification_models = {
    "Most-frequent baseline": dummy_classifier,
    "Logistic regression": logistic_search.best_estimator_,
    "Random forest classifier": classifier_forest_search.best_estimator_,
}

classification_results = pd.DataFrame(
    [
        classification_metrics(
            "Most-frequent baseline", dummy_classifier, X_test, y_test_cls
        ),
        classification_metrics(
            "Logistic regression", logistic_search.best_estimator_, X_test, y_test_cls,
            logistic_search.best_score_,
        ),
        classification_metrics(
            "Random forest classifier", classifier_forest_search.best_estimator_, X_test, y_test_cls,
            classifier_forest_search.best_score_,
        ),
    ]
).sort_values("test_f1", ascending=False, ignore_index=True)

classification_results


In [ ]:
classification_tuning = pd.DataFrame(
    {
        "model": ["Logistic regression", "Random forest classifier"],
        "best_grouped_cv_f1": [logistic_search.best_score_, classifier_forest_search.best_score_],
        "best_parameters": [logistic_search.best_params_, classifier_forest_search.best_params_],
    }
)

best_classifier_name = classification_tuning.sort_values(
    "best_grouped_cv_f1", ascending=False
).iloc[0]["model"]
best_classifier = classification_models[best_classifier_name]
classification_prediction = best_classifier.predict(X_test)

matrix = confusion_matrix(y_test_cls, classification_prediction)
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(matrix, cmap="Blues")
for row_index in range(matrix.shape[0]):
    for column_index in range(matrix.shape[1]):
        ax.text(column_index, row_index, matrix[row_index, column_index], ha="center", va="center")
ax.set(
    title=f"Confusion matrix: {best_classifier_name}",
    xlabel="Predicted class",
    ylabel="Actual class",
    xticks=[0, 1],
    yticks=[0, 1],
    xticklabels=["No decline", "Decline"],
    yticklabels=["No decline", "Decline"],
)
plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print("Selected from grouped cross-validation:", best_classifier_name)
classification_tuning


## 17. Compare performance with the baselines

Model selection and final evaluation answer different questions. Grouped cross-validation selects hyperparameters without using the final holdout. The later-year holdout then tests whether that selection generalizes through time.

If a tuned model does not beat persistence in regression, or does not materially exceed 0.50 balanced accuracy in classification, the correct conclusion is that these coarse annual national indicators do not yet provide strong short-horizon predictive information. Hyperparameters should not be retuned to make the holdout result look better.


In [ ]:
baseline_regression_mae = regression_results.set_index("model").loc[
    "Persistence baseline", "test_mae"
]
selected_regression_mae = regression_results.set_index("model").loc[
    best_regression_name, "test_mae"
]
regression_improvement_pct = (
    (baseline_regression_mae - selected_regression_mae)
    / baseline_regression_mae
    * 100
)

baseline_balanced_accuracy = classification_results.set_index("model").loc[
    "Most-frequent baseline", "test_balanced_accuracy"
]
best_holdout_classifier = classification_results.sort_values(
    "test_balanced_accuracy", ascending=False
).iloc[0]

baseline_comparison = pd.DataFrame(
    {
        "question": [
            "Selected regression vs persistence",
            "Best classification balanced accuracy vs baseline",
        ],
        "baseline": [baseline_regression_mae, baseline_balanced_accuracy],
        "model_result": [
            selected_regression_mae,
            best_holdout_classifier["test_balanced_accuracy"],
        ],
        "difference": [
            regression_improvement_pct,
            best_holdout_classifier["test_balanced_accuracy"]
            - baseline_balanced_accuracy,
        ],
        "difference_unit": ["MAE improvement (%)", "balanced-accuracy points"],
    }
)

if regression_improvement_pct > 0:
    print(
        f"{best_regression_name} improves holdout MAE over persistence by "
        f"{regression_improvement_pct:.1f}%."
    )
else:
    print(
        f"{best_regression_name} does not beat persistence on the time holdout; "
        f"its MAE is {-regression_improvement_pct:.1f}% higher."
    )

print(
    f"The strongest holdout balanced accuracy is "
    f"{best_holdout_classifier['test_balanced_accuracy']:.2f} "
    f"from {best_holdout_classifier['model']}."
)

baseline_comparison


## 18. Interpretation guardrails

This project predicts a statistical bird index, not individual birds or extinction. Several limitations should appear in the final presentation:

1. The country-year sample is small after combining all indicators.
2. National indices and environmental indicators may use different collection methods and revision schedules.
3. Pesticide sales are a proxy for pressure, not direct exposure of bird populations.
4. Protected-area coverage measures area, not ecological quality or farmland-specific protection.
5. Country-level associations can differ from relationships at farm or habitat level.
6. Predictive importance does not establish causality.
7. A model that fails to beat persistence is still an informative result: the environmental data may be too coarse, delayed, or incomplete for short-horizon prediction.

The strongest portfolio presentation will explicitly compare training, grouped-CV, and time-holdout performance and discuss the bias–variance tradeoff suggested by their differences.


## 19. Export the modelling data and results

The fitted models are serialized with `pickle`. The output should only be loaded in a trusted Python environment. The exported CSV files provide a transparent record of the training data, predictions, tuning results, metrics, and feature importance.


In [ ]:
import pickle


model_output_dir = Path("outputs/machine_learning")
model_output_dir.mkdir(parents=True, exist_ok=True)

ml_exports = {
    "organic_farming.csv": organic_farming,
    "pesticide_intensity.csv": pesticide_intensity,
    "bird_ml_panel_all_rows.csv": ml_panel,
    "bird_ml_model_data.csv": model_data,
    "regression_results.csv": regression_results,
    "regression_tuning.csv": regression_tuning,
    "regression_test_predictions.csv": test_predictions,
    "regression_feature_importance.csv": regression_feature_importance,
    "classification_results.csv": classification_results,
    "classification_tuning.csv": classification_tuning,
    "baseline_comparison.csv": baseline_comparison,
}

for filename, dataframe in ml_exports.items():
    dataframe.to_csv(model_output_dir / filename, index=False)

with (model_output_dir / "best_regression_model.pkl").open("wb") as file:
    pickle.dump(best_regression_model, file)

with (model_output_dir / "best_classifier.pkl").open("wb") as file:
    pickle.dump(best_classifier, file)

ml_export_summary = pd.DataFrame(
    {
        "file": [*ml_exports.keys(), "best_regression_model.pkl", "best_classifier.pkl"],
        "type": ["CSV"] * len(ml_exports) + ["Python pickle", "Python pickle"],
        "rows": [len(dataframe) for dataframe in ml_exports.values()] + [pd.NA, pd.NA],
    }
)

ml_export_summary
